# XGBoost Volatility Backtest

CPU-only tree-based walk-forward backtest. XGBoost handles NaN natively,
so no scaling or missing-value imputation is needed for features.

In [ ]:
# ── Setup: clone repo, install deps ──
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q numpy pandas scikit-learn pyarrow tqdm xgboost

import sys  # noqa: E402

sys.path.insert(0, "/content/harxhar/colab")

In [ ]:
# ── Parameters ──
HORIZON = 1
TRAIN_WINDOW_DAYS = 500
PERIODS_PER_DAY = 48
REFIT_FREQUENCY = 5
DATA_PATH = "all30min"

In [ ]:
# export
"""XGBoost volatility backtest executor."""

import argparse
import json
import os

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

from src.evaluation import apply_duan_smearing
from src.loading import load_raw_data
from src.scaling import run_backtest
from src.transforms import (
    PERIODS_PER_DAY,
    add_calendar_features,
    apply_horizon_shift,
    generate_har_features,
    resolve_har_lags,
    robust_transform,
)


In [ ]:
# ── Imports smoke test ──
# Confirm every symbol we just brought into scope is bound and usable.
_expected = [
    "argparse", "json", "os", "np", "pd", "XGBRegressor",
    "apply_duan_smearing", "load_raw_data", "run_backtest",
    "PERIODS_PER_DAY", "add_calendar_features", "apply_horizon_shift",
    "generate_har_features", "resolve_har_lags", "robust_transform",
]
_missing = [n for n in _expected if n not in dir()]
assert not _missing, f"missing imports: {_missing}"
print(f"imports OK ({len(_expected)} symbols), PERIODS_PER_DAY={PERIODS_PER_DAY}")


In [ ]:
# ── Load + transform ──
import numpy as np

from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data(DATA_PATH, allow_missing=True)
print(f"Loaded: {df.shape}")

# Full transform on RV target (diurnal + winsor)
adj_rv, baseline = robust_transform(df, "RV", is_target=True, use_diurnal=True, winsor_window=240)
df["adj_RV"] = adj_rv
df["baseline"] = baseline

print(f"adj_RV stats:\n{df['adj_RV'].describe()}")

In [ ]:
# --- Features from shared pipeline ---
from src.transforms import resolve_har_lags, generate_har_features, add_calendar_features

df, har_names = generate_har_features(df, target_col="adj_RV")
cal_names = add_calendar_features(df)
feature_names = har_names + cal_names

print(f"HAR lags: {resolve_har_lags()}")
print(f"Features ({len(feature_names)}): {feature_names}")

# Drop initial NaN rows from HAR computation
max_lag = resolve_har_lags()[-1]
df = df.iloc[max_lag:].reset_index(drop=True)
print(f"Shape after lag trim: {df.shape}")

In [ ]:
# --- XGBoost walk-forward backtest ---
from xgboost import XGBRegressor
from src.transforms import apply_horizon_shift
from src.scaling import run_backtest

X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)
dates = df["t"]
baselines = df["baseline"].values.astype(np.float64)

# Horizon shift
X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, HORIZON)

train_win = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY
print(f"Samples: {len(y)}, Features: {X.shape[1]}, Train window: {train_win}")

model_fn = lambda: XGBRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.1,
    tree_method="hist", n_jobs=-1,
)

predictions = run_backtest(model_fn, X, y, train_win=train_win, refit_frequency=REFIT_FREQUENCY, use_scaling=False)
print(f"Predictions: {predictions.shape}, NaN count: {np.isnan(predictions).sum()}")

# Fit final model on last window for feature importance plot
model = model_fn()
model.fit(X[-train_win:], y[-train_win:])

In [ ]:
# ── Feature importance ──
import matplotlib.pyplot as plt

importance = model.feature_importances_
sorted_idx = np.argsort(importance)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([feature_names[i] for i in sorted_idx], importance[sorted_idx])
ax.set_xlabel("Feature Importance (gain)")
ax.set_title("XGBoost Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation ---
from src.evaluation import apply_duan_smearing, calculate_metrics
import pandas as pd

oos_start = train_win
y_oos = y[oos_start:]
dates_oos = dates.iloc[oos_start:].values
baselines_oos = baselines[oos_start:]

pred_raw, true_raw = apply_duan_smearing(predictions, y_oos, baselines_oos)

results_df = pd.DataFrame({
    "date": dates_oos, "horizon": HORIZON,
    "true_adj": y_oos, "pred_adj": predictions,
    "true_raw": true_raw, "pred_raw": pred_raw,
})

metrics = calculate_metrics(results_df)
print("\n".join(f"{k:>10s}: {v:.6f}" for k, v in metrics.items()))

In [ ]:
# export
def main() -> None:
    parser = argparse.ArgumentParser(description="XGBoost walk-forward backtest")
    parser.add_argument("--data-path", default="all30min")
    parser.add_argument("--horizon", type=int, default=1)
    parser.add_argument("--train-window", type=int, default=500, help="training window in days")
    parser.add_argument(
        "--refit-frequency",
        type=int,
        default=1,
        help="how often to refit the model during walk-forward (1 = every step)",
    )
    parser.add_argument("--start", type=int, default=0)
    parser.add_argument("--end", type=int, default=-1)
    parser.add_argument("--output-file", required=True)
    parser.add_argument("--params-file", default=None, help="JSON file with tuned hyperparams")
    args = parser.parse_args()

    tuned_params = {}
    if args.params_file:
        with open(args.params_file) as f:
            tuned_params = json.load(f)

    train_win_periods = args.train_window * PERIODS_PER_DAY

    df = load_raw_data(args.data_path, allow_missing=True)
    adj_rv, baseline = robust_transform(df, "RV", is_target=True, use_diurnal=True, winsor_window=240)
    df["adj_RV"] = adj_rv
    df["baseline"] = baseline

    df, har_names = generate_har_features(df, target_col="adj_RV")
    cal_names = add_calendar_features(df)
    feature_names = har_names + cal_names

    max_lag = resolve_har_lags()[-1]
    df = df.iloc[max_lag:].reset_index(drop=True)

    X = df[feature_names].values.astype(np.float64)
    y = df["adj_RV"].values.astype(np.float64)
    dates = df["t"]
    baselines = df["baseline"].values.astype(np.float64)

    X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, args.horizon)

    start = args.start
    end = len(X) if args.end == -1 else args.end
    X_chunk, y_chunk = X[start:end], y[start:end]
    dates_chunk = dates.iloc[start:end].reset_index(drop=True)
    baselines_chunk = baselines[start:end]

    if train_win_periods >= len(X_chunk):
        raise ValueError(f"train_window ({train_win_periods}) >= chunk size ({len(X_chunk)})")

    def model_fn():
        defaults = dict(n_estimators=500, max_depth=5, learning_rate=0.1, tree_method="hist", n_jobs=-1)
        defaults.update(tuned_params)
        return XGBRegressor(**defaults)

    preds = run_backtest(
        model_fn,
        X_chunk,
        y_chunk,
        train_win=train_win_periods,
        refit_frequency=args.refit_frequency,
        use_scaling=False,
    )

    oos_start = train_win_periods
    y_oos = y_chunk[oos_start:]
    dates_oos = dates_chunk.iloc[oos_start:].values
    baselines_oos = baselines_chunk[oos_start:]

    pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

    results = pd.DataFrame(
        {
            "date": dates_oos,
            "horizon": args.horizon,
            "true_adj": y_oos,
            "pred_adj": preds,
            "true_raw": true_raw,
            "pred_raw": pred_raw,
        }
    )

    os.makedirs(os.path.dirname(args.output_file) or ".", exist_ok=True)
    results.to_csv(args.output_file, index=False)
    from src.evaluation import save_chunk_reduce

    save_chunk_reduce(results, args.output_file)
    print(f"Saved {len(results)} rows -> {args.output_file}")


In [ ]:
# ── Smoke-test main() via argparse ──
# Call main() with a tiny --output-file; the notebook has already exercised the
# same pipeline (load, transform, features, backtest, smearing) interactively
# above, so we just confirm the argparse-wired entrypoint is callable.
import inspect

assert callable(main), "main must be defined"
sig = inspect.signature(main)
assert list(sig.parameters) == [], f"main() should take no args; got {sig}"
print("main signature OK:", sig)


In [ ]:
# export
if __name__ == "__main__":
    main()


In [ ]:
# ── Guard sanity check ──
# The `if __name__ == "__main__": main()` block only runs when the module is
# executed directly; in the notebook __name__ != "__main__", so importing or
# re-running this file as a script is the only way to trigger it. Nothing to
# exercise here beyond confirming main is still bound.
assert "main" in dir(), "main() must be in scope for the guard to resolve"
print("guard will dispatch to:", main)


In [ ]:
from _exporter import export_notebook
export_notebook("ml_xgboost.ipynb", "../src/ml_xgboost.py")
